In [1]:
import pandas as pd

In [2]:
aq_orig = pd.read_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Visual Analytics/Team Project/Processed data/nyc_air_clean_by_period_place.csv')
aq_new  = pd.read_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Visual Analytics/Team Project/Raw data/Air Quality/ad_viz_tile_data 2005-2025.csv')

In [3]:
print("Original shape:", aq_orig.shape)
print("New data shape:", aq_new.shape)

print("\nOriginal columns:")
print(aq_orig.columns)

print("\nNew file columns:")
print(aq_new.columns)

Original shape: (18862, 8)
New data shape: (7622, 6)

Original columns:
Index(['Geo Join ID', 'Geo Place Name', 'Geo Type Name', 'Start_Date',
       'Time Period', 'Name', 'Measure Info', 'Data Value'],
      dtype='object')

New file columns:
Index(['Date', ' AQI Value', 'Main Pollutant', 'Site Name', 'Site ID',
       'Source'],
      dtype='object')


In [4]:
aq_orig.columns = aq_orig.columns.str.strip()
aq_new.columns  = aq_new.columns.str.strip()


In [5]:
print("\nSample of processed data:")
display(aq_orig.head())

print("\nUnique 'Time Period' values (first 20):")
print(aq_orig['Time Period'].unique()[:20])

print("\nUnique 'Name' values:")
print(aq_orig['Name'].unique())

print("\nUnique 'Measure Info' values:")
print(aq_orig['Measure Info'].unique())

print("\nUnique 'Geo Type Name' values:")
print(aq_orig['Geo Type Name'].unique())

print("\nUnique 'Geo Place Name' values (first 20):")
print(aq_orig['Geo Place Name'].unique()[:20])


Sample of processed data:


,Geo Join ID,Geo Place Name,Geo Type Name,Start_Date,Time Period,Name,Measure Info,Data Value
0,310,Bay Ridge and Dyker Heights (CD10),CD,2005-01-01,2005,Annual vehicle miles traveled,per square mile,58.440812
1,310,Bay Ridge and Dyker Heights (CD10),CD,2005-01-01,2005,Annual vehicle miles traveled (cars),per square mile,54.575696
2,310,Bay Ridge and Dyker Heights (CD10),CD,2005-01-01,2005,Annual vehicle miles traveled (trucks),per square mile,3.393163
3,404,Bayside - Little Neck,UHF42,2005-01-01,2005,Annual vehicle miles traveled,per square mile,60.492651
4,404,Bayside - Little Neck,UHF42,2005-01-01,2005,Outdoor Air Toxics - Benzene,Âµg/m3,1.900000



Unique 'Time Period' values (first 20):
['2005' '2005-2007' '2009-2011' '2010' '2011' '2012-2014' '2013' '2014'
 '2015' '2015-2017' '2017-2019' '2019' 'Annual Average 2009'
 'Annual Average 2010' 'Annual Average 2011' 'Annual Average 2012'
 'Annual Average 2013' 'Annual Average 2014' 'Annual Average 2015'
 'Annual Average 2016']

Unique 'Name' values:
['Annual vehicle miles traveled' 'Annual vehicle miles traveled (cars)'
 'Annual vehicle miles traveled (trucks)' 'Outdoor Air Toxics - Benzene'
 'Outdoor Air Toxics - Formaldehyde'
 'Cardiac and respiratory deaths due to Ozone'
 'Asthma emergency department visits due to PM2.5'
 'Asthma hospitalizations due to Ozone' 'Deaths due to PM2.5'
 'Respiratory hospitalizations due to PM2.5 (age 20+)'
 'Asthma emergency departments visits due to Ozone'
 'Cardiovascular hospitalizations due to PM2.5 (age 40+)'
 'Boiler Emissions- Total NOx Emissions'
 'Boiler Emissions- Total PM2.5 Emissions'
 'Boiler Emissions- Total SO2 Emissions' 'Fine particl

In [6]:
# Parse date
aq_new['Date'] = pd.to_datetime(aq_new['Date'], errors='coerce')
aq_new = aq_new.dropna(subset=['Date'])

aq_new['Year'] = aq_new['Date'].dt.year

print("Years present in raw data:", sorted(aq_new['Year'].unique()))


Years present in raw data: [np.int32(2005), np.int32(2006), np.int32(2007), np.int32(2008), np.int32(2009), np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]


In [7]:
aq_new_24_25 = aq_new[aq_new['Year'].isin([2024, 2025])].copy()
print("Raw rows for 2024–2025:", aq_new_24_25.shape)
display(aq_new_24_25.head())

Raw rows for 2024–2025: (683, 7)


,Date,AQI Value,Main Pollutant,Site Name,Site ID,Source,Year
6939,2024-01-01,58,PM2.5,Intermediate School 143,36-061-0115,AQS,2024
6940,2024-01-02,54,PM2.5,Intermediate School 143,36-061-0115,AQS,2024
6941,2024-01-03,53,PM2.5,Intermediate School 143,36-061-0115,AQS,2024
6942,2024-01-04,51,PM2.5,CCNY,36-061-0135,AQS,2024
6943,2024-01-05,51,PM2.5,Intermediate School 143,36-061-0115,AQS,2024


In [8]:
# Aggregate to yearly average AQI across all sites for each year
aq_agg_24_25 = (
    aq_new_24_25
    .groupby('Year', as_index=False)['AQI Value']
    .mean()
    .rename(columns={'AQI Value': 'Data Value'})
)

print("\nYearly citywide average AQI for 2024–2025:")
display(aq_agg_24_25)


Yearly citywide average AQI for 2024–2025:


,Year,Data Value
0,2024,46.830601
1,2025,46.053628


In [9]:
# OPTIONAL: per pollutant per year (comment out if you don't need this)
aq_agg_24_25 = (
    aq_new_24_25
    .groupby(['Year', 'Main Pollutant'], as_index=False)['AQI Value']
    .mean()
    .rename(columns={'AQI Value': 'Data Value'})
)

display(aq_agg_24_25.head())

,Year,Main Pollutant,Data Value
0,2024,Ozone,46.511450
1,2024,PM2.5,47.008511
2,2025,Ozone,44.298611
3,2025,PM2.5,47.514451


In [10]:
orig = aq_orig.copy()

# Try to find a row where Name looks like it’s AQI-related
mask_aqi = orig['Name'].astype(str).str.contains('AQI', case=False, na=False)

if mask_aqi.any():
    ref_row = orig[mask_aqi].iloc[0]
    print("Using this row as template for new AQI records:")
else:
    ref_row = orig.iloc[0]
    print("No explicit 'AQI' in Name; using first row as template:")
    
display(ref_row)

No explicit 'AQI' in Name; using first row as template:


Geo Join ID                                      310
Geo Place Name    Bay Ridge and Dyker Heights (CD10)
Geo Type Name                                     CD
Start_Date                                2005-01-01
Time Period                                     2005
Name                   Annual vehicle miles traveled
Measure Info                         per square mile
Data Value                                 58.440812
Name: 0, dtype: object

In [11]:
geo_join_id   = ref_row['Geo Join ID']
geo_place     = ref_row['Geo Place Name']
geo_type      = ref_row['Geo Type Name']
name_value    = ref_row['Name']          # Name column for AQI metric
measure_value = ref_row['Measure Info']  # Measurement descriptor

In [12]:
aq_new_proc = aq_agg_24_25.copy()

# If you used pollutant-specific aggregation and want Name to depend on it,
# you could dynamically build Name here. For now we keep Name from template.

aq_new_proc['Geo Join ID']   = geo_join_id
aq_new_proc['Geo Place Name'] = geo_place
aq_new_proc['Geo Type Name']  = geo_type
aq_new_proc['Name']           = name_value
aq_new_proc['Measure Info']   = measure_value

# Create Start_Date as 'YYYY-01-01' and Time Period as string year
aq_new_proc['Start_Date'] = pd.to_datetime(aq_new_proc['Year'].astype(str) + '-01-01')
aq_new_proc['Time Period'] = aq_new_proc['Year'].astype(str)

# Ensure column order matches original
desired_cols = [
    'Geo Join ID',
    'Geo Place Name',
    'Geo Type Name',
    'Start_Date',
    'Time Period',
    'Name',
    'Measure Info',
    'Data Value'
]

aq_new_proc = aq_new_proc[desired_cols]

print("\nNew processed-style rows for 2024–25:")
display(aq_new_proc)



New processed-style rows for 2024–25:


,Geo Join ID,Geo Place Name,Geo Type Name,Start_Date,Time Period,Name,Measure Info,Data Value
0,310,Bay Ridge and Dyker Heights (CD10),CD,2024-01-01,2024,Annual vehicle miles traveled,per square mile,46.511450
1,310,Bay Ridge and Dyker Heights (CD10),CD,2024-01-01,2024,Annual vehicle miles traveled,per square mile,47.008511
2,310,Bay Ridge and Dyker Heights (CD10),CD,2025-01-01,2025,Annual vehicle miles traveled,per square mile,44.298611
3,310,Bay Ridge and Dyker Heights (CD10),CD,2025-01-01,2025,Annual vehicle miles traveled,per square mile,47.514451


In [13]:
aq_orig = aq_orig.copy()

# Make sure Time Period is string
aq_orig['Time Period'] = aq_orig['Time Period'].astype(str)

# Filter out any existing 2024–25 rows just in case
aq_orig_clean = aq_orig[~aq_orig['Time Period'].isin(['2024', '2025'])].copy()

print("Original rows before filter:", aq_orig.shape[0])
print("Original rows after  filter:", aq_orig_clean.shape[0])

Original rows before filter: 18862
Original rows after  filter: 18862


In [14]:
aq_combined = pd.concat([aq_orig_clean, aq_new_proc], ignore_index=True)

print("Final combined shape:", aq_combined.shape)

Final combined shape: (18866, 8)


In [15]:
print("\nCounts by Time Period:")
print(aq_combined['Time Period'].value_counts().sort_index())


Counts by Time Period:
Time Period
2005                   417
2005-2007              480
2009-2011              480
2010                   321
2011                   214
2012-2014              480
2013                   144
2014                    96
2015                   144
2015-2017              480
2017-2019              480
2019                   321
2024                     2
2025                     2
Annual Average 2009    282
Annual Average 2010    282
Annual Average 2011    282
Annual Average 2012    282
Annual Average 2013    282
Annual Average 2014    282
Annual Average 2015    282
Annual Average 2016    282
Annual Average 2017    282
Annual Average 2018    282
Annual Average 2019    282
Annual Average 2020    282
Annual Average 2021    282
Annual Average 2022    282
Annual Average 2023    282
Summer 2009            423
Summer 2010            423
Summer 2011            423
Summer 2012            423
Summer 2013            423
Summer 2014            423
Summer 2015        

In [16]:
# Drop the previous '2024' and '2025' rows (wrong format)
mask_bad = aq_combined['Time Period'].isin(['2024', '2025'])
print("Rows to drop (old patch):", mask_bad.sum())

aq_combined = aq_combined[~mask_bad].copy()
print("Shape after dropping old 2024/25 rows:", aq_combined.shape)

Rows to drop (old patch): 4
Shape after dropping old 2024/25 rows: (18862, 8)


In [17]:
import pandas as pd

# If not done already:
aq_new['Date'] = pd.to_datetime(aq_new['Date'], errors='coerce')
aq_new = aq_new.dropna(subset=['Date'])
aq_new['Year'] = aq_new['Date'].dt.year

# Filter 2024 & 2025
aq_new_24_25 = aq_new[aq_new['Year'].isin([2024, 2025])].copy()
print("Raw rows for 2024–2025:", aq_new_24_25.shape)

# Citywide annual average AQI per year
annual_citywide = (
    aq_new_24_25
    .groupby('Year', as_index=False)['AQI Value']
    .mean()
    .rename(columns={'AQI Value': 'Data Value'})
)

print("\nCitywide Annual Average AQI for 2024/25:")
display(annual_citywide)


Raw rows for 2024–2025: (683, 7)

Citywide Annual Average AQI for 2024/25:


,Year,Data Value
0,2024,46.830601
1,2025,46.053628


In [18]:
# Use any recent complete "Annual Average" year as a template, e.g., 2023
mask_aa_2023 = aq_combined['Time Period'] == 'Annual Average 2023'
geo_template = aq_combined.loc[mask_aa_2023, ['Geo Join ID', 'Geo Place Name', 'Geo Type Name']].drop_duplicates()

print("Number of geos in template:", len(geo_template))
display(geo_template.head())


Number of geos in template: 141


,Geo Join ID,Geo Place Name,Geo Type Name
8005,310,Bay Ridge and Dyker Heights (CD10),CD
8007,404,Bayside - Little Neck,UHF42
8009,404406,Bayside Little Neck-Fresh Meadows,UHF34
8011,411,Bayside and Little Neck (CD11),CD
8013,303,Bedford Stuyvesant (CD3),CD


In [19]:
# Take one representative row for the Annual Average metric (e.g., 2023)
ref_row = aq_combined[mask_aa_2023].iloc[0]

name_value    = ref_row['Name']
measure_value = ref_row['Measure Info']

print("Using Name:", name_value)
print("Using Measure Info:", measure_value)


Using Name: Nitrogen dioxide (NO2)
Using Measure Info: ppb


In [20]:
# Prepare citywide annual averages for cross join
annual_citywide['Time Period'] = 'Annual Average ' + annual_citywide['Year'].astype(str)
annual_citywide['Start_Date']  = pd.to_datetime(annual_citywide['Year'].astype(str) + '-01-01')

# Add Name and Measure Info
annual_citywide['Name']         = name_value
annual_citywide['Measure Info'] = measure_value

# For cross join: add a dummy key in both tables
geo_template['_key']        = 1
annual_citywide['_key']     = 1

new_aa_rows = geo_template.merge(annual_citywide, on='_key').drop(columns=['_key'])

# Reorder columns to match original
new_aa_rows = new_aa_rows[[
    'Geo Join ID',
    'Geo Place Name',
    'Geo Type Name',
    'Start_Date',
    'Time Period',
    'Name',
    'Measure Info',
    'Data Value'
]]

print("New Annual Average 2024/25 rows shape:", new_aa_rows.shape)
print(new_aa_rows['Time Period'].value_counts())
display(new_aa_rows.head())


New Annual Average 2024/25 rows shape: (282, 8)
Time Period
Annual Average 2024    141
Annual Average 2025    141
Name: count, dtype: int64


,Geo Join ID,Geo Place Name,Geo Type Name,Start_Date,Time Period,Name,Measure Info,Data Value
0,310,Bay Ridge and Dyker Heights (CD10),CD,2024-01-01,Annual Average 2024,Nitrogen dioxide (NO2),ppb,46.830601
1,310,Bay Ridge and Dyker Heights (CD10),CD,2025-01-01,Annual Average 2025,Nitrogen dioxide (NO2),ppb,46.053628
2,404,Bayside - Little Neck,UHF42,2024-01-01,Annual Average 2024,Nitrogen dioxide (NO2),ppb,46.830601
3,404,Bayside - Little Neck,UHF42,2025-01-01,Annual Average 2025,Nitrogen dioxide (NO2),ppb,46.053628
4,404406,Bayside Little Neck-Fresh Meadows,UHF34,2024-01-01,Annual Average 2024,Nitrogen dioxide (NO2),ppb,46.830601


In [21]:
aq_combined = pd.concat([aq_combined, new_aa_rows], ignore_index=True)

print("Final combined shape:", aq_combined.shape)

print("\nCounts by Time Period (sorted):")
print(aq_combined['Time Period'].value_counts().sort_index())


Final combined shape: (19144, 8)

Counts by Time Period (sorted):
Time Period
2005                   417
2005-2007              480
2009-2011              480
2010                   321
2011                   214
2012-2014              480
2013                   144
2014                    96
2015                   144
2015-2017              480
2017-2019              480
2019                   321
Annual Average 2009    282
Annual Average 2010    282
Annual Average 2011    282
Annual Average 2012    282
Annual Average 2013    282
Annual Average 2014    282
Annual Average 2015    282
Annual Average 2016    282
Annual Average 2017    282
Annual Average 2018    282
Annual Average 2019    282
Annual Average 2020    282
Annual Average 2021    282
Annual Average 2022    282
Annual Average 2023    282
Annual Average 2024    141
Annual Average 2025    141
Summer 2009            423
Summer 2010            423
Summer 2011            423
Summer 2012            423
Summer 2013            423
Summ

In [23]:
import numpy as np

# ==============================
# Sanity Check for AQI 2024–2025
# ==============================

# ---------- 1. Prepare processed Annual Average series ----------

# Filter only "Annual Average ..." rows
aa = aq_combined[aq_combined['Time Period'].astype(str).str.startswith('Annual Average')].copy()

# Extract Year from "Annual Average YYYY"
aa['Year'] = aa['Time Period'].str.extract(r'(\d{4})').astype(int)

# Citywide mean per year (average across all geos in that year)
aa_yearly = (
    aa.groupby('Year', as_index=False)['Data Value']
      .mean()
      .rename(columns={'Data Value': 'proc_mean_aqi'})
)

print("=== Processed: Citywide Annual Average AQI per Year ===")
print(aa_yearly)

# ---------- 2. Z-score sanity check for 2024 & 2025 vs <= 2023 ----------

baseline = aa_yearly[aa_yearly['Year'] <= 2023]['proc_mean_aqi']
mean_base = baseline.mean()
std_base  = baseline.std()

aa_yearly['z_from_baseline'] = (aa_yearly['proc_mean_aqi'] - mean_base) / std_base

print("\n=== Z-scores of Annual Averages vs Baseline (<= 2023) ===")
print(aa_yearly)

print("\nZ-scores specifically for 2024 & 2025:")
print(aa_yearly[aa_yearly['Year'].isin([2024, 2025])])

# ---------- 3. Prepare raw annual citywide averages (overlap years) ----------

# Make sure Date exists and is parsed in raw
aq_new = aq_new.copy()
aq_new['Date'] = pd.to_datetime(aq_new['Date'], errors='coerce')
aq_new = aq_new.dropna(subset=['Date'])
aq_new['Year'] = aq_new['Date'].dt.year

# Define overlap range where processed has Annual Averages
start_year = 2009
end_year   = 2023

raw_aa_overlap = (
    aq_new[aq_new['Year'].between(start_year, end_year)]
      .groupby('Year', as_index=False)['AQI Value']
      .mean()
      .rename(columns={'AQI Value': 'raw_mean_aqi'})
)

print(f"\n=== Raw: Citywide Annual Mean AQI for {start_year}–{end_year} ===")
print(raw_aa_overlap)

# ---------- 4. Compare processed vs raw for overlap years ----------

proc_aa_overlap = aa_yearly[aa_yearly['Year'].between(start_year, end_year)].copy()

compare = raw_aa_overlap.merge(proc_aa_overlap, on='Year', how='inner')
compare['abs_diff'] = (compare['proc_mean_aqi'] - compare['raw_mean_aqi']).abs()
compare['pct_diff'] = 100 * compare['abs_diff'] / compare['proc_mean_aqi'].replace(0, np.nan)

print("\n=== Comparison: Processed vs Raw Annual Mean AQI (Overlap Years) ===")
print(compare)

print("\nSummary of % differences (overlap years):")
print(compare['pct_diff'].describe())

# ---------- 5. Extra quick checks: last overlap year (2023) & new years (raw) ----------

print("\n=== 2023 Only: Raw vs Processed ===")
print(compare[compare['Year'] == 2023])

# Raw citywide means for 2024 & 2025, for context
raw_24_25 = (
    aq_new[aq_new['Year'].isin([2024, 2025])]
      .groupby('Year', as_index=False)['AQI Value']
      .mean()
      .rename(columns={'AQI Value': 'raw_mean_aqi'})
)

print("\n=== Raw: Citywide Annual Mean AQI for 2024 & 2025 ===")
print(raw_24_25)

print("\n=== Final sanity view: Processed vs Baseline z-scores (focus 2024–2025) ===")
print(aa_yearly)

=== Processed: Citywide Annual Average AQI per Year ===
    Year  proc_mean_aqi
0   2009      18.363369
1   2010      16.914929
2   2011      17.377234
3   2012      15.745745
4   2013      15.222979
5   2014      15.206383
6   2015      14.696335
7   2016      13.678298
8   2017      13.711312
9   2018      13.103546
10  2019      12.166596
11  2020      11.567340
12  2021      11.882687
13  2022      11.062262
14  2023      11.787304
15  2024      46.830601
16  2025      46.053628

=== Z-scores of Annual Averages vs Baseline (<= 2023) ===
    Year  proc_mean_aqi  z_from_baseline
0   2009      18.363369         1.834587
1   2010      16.914929         1.201540
2   2011      17.377234         1.403593
3   2012      15.745745         0.690542
4   2013      15.222979         0.462065
5   2014      15.206383         0.454812
6   2015      14.696335         0.231893
7   2016      13.678298        -0.213045
8   2017      13.711312        -0.198616
9   2018      13.103546        -0.464243
10

In [24]:
import pandas as pd
import numpy as np

# ==============================
# FIX 2024–2025 ANNUAL AVERAGES
# ==============================

# 1) Prepare processed Annual Average series (all years)

aa = aq_combined[aq_combined['Time Period'].astype(str).str.startswith('Annual Average')].copy()
aa['Year'] = aa['Time Period'].str.extract(r'(\d{4})').astype(int)

aa_yearly = (
    aa.groupby('Year', as_index=False)['Data Value']
      .mean()
      .rename(columns={'Data Value': 'proc_mean_aqi'})
)

print("Processed (citywide) Annual Average AQI per year:")
print(aa_yearly)

# 2) Prepare raw annual citywide means

aq_new = aq_new.copy()
aq_new['Date'] = pd.to_datetime(aq_new['Date'], errors='coerce')
aq_new = aq_new.dropna(subset=['Date'])
aq_new['Year'] = aq_new['Date'].dt.year

raw_yearly = (
    aq_new.groupby('Year', as_index=False)['AQI Value']
          .mean()
          .rename(columns={'AQI Value': 'raw_mean_aqi'})
)

print("\nRaw citywide annual mean AQI per year:")
print(raw_yearly)

# 3) Build mapping raw_mean_aqi -> proc_mean_aqi using overlap years 2009–2023

start_year = 2009
end_year   = 2023

overlap_raw  = raw_yearly[raw_yearly['Year'].between(start_year, end_year)]
overlap_proc = aa_yearly[aa_yearly['Year'].between(start_year, end_year)]

mapping_df = overlap_raw.merge(overlap_proc, on='Year', how='inner')

print("\nOverlap years used for mapping:")
print(mapping_df)

# Fit simple linear regression: proc ≈ m * raw + c
m, c = np.polyfit(mapping_df['raw_mean_aqi'], mapping_df['proc_mean_aqi'], 1)
print(f"\nFitted mapping: proc_mean ≈ {m:.4f} * raw_mean + {c:.4f}")

# 4) Predict "processed-style" values for 2024 & 2025 from raw means

raw_24_25 = raw_yearly[raw_yearly['Year'].isin([2024, 2025])].copy()
raw_24_25['pred_proc_mean_aqi'] = m * raw_24_25['raw_mean_aqi'] + c

print("\nRaw and predicted processed means for 2024 & 2025:")
print(raw_24_25)

# 5) Update aq_combined:
#    For each year (2024, 2025), replace Data Value in "Annual Average YYYY" rows
#    so that the citywide mean matches the predicted value.

aq_combined = aq_combined.copy()

for _, row in raw_24_25.iterrows():
    year = int(row['Year'])
    target_period = f"Annual Average {year}"
    pred_value = float(row['pred_proc_mean_aqi'])

    mask = aq_combined['Time Period'] == target_period

    # Current mean before change
    current_mean = aq_combined.loc[mask, 'Data Value'].mean()
    print(f"\nYear {year}: current mean={current_mean:.4f}, replacing with {pred_value:.4f}")

    # Set all geos for that year to the predicted value
    aq_combined.loc[mask, 'Data Value'] = pred_value

# 6) Recompute processed annual means and z-scores to sanity-check

aa_fixed = aq_combined[aq_combined['Time Period'].astype(str).str.startswith('Annual Average')].copy()
aa_fixed['Year'] = aa_fixed['Time Period'].str.extract(r'(\d{4})').astype(int)

aa_fixed_yearly = (
    aa_fixed.groupby('Year', as_index=False)['Data Value']
            .mean()
            .rename(columns={'Data Value': 'proc_mean_aqi_fixed'})
)

# Baseline on <= 2023
baseline = aa_fixed_yearly[aa_fixed_yearly['Year'] <= 2023]['proc_mean_aqi_fixed']
mean_base = baseline.mean()
std_base  = baseline.std()

aa_fixed_yearly['z_from_baseline'] = (aa_fixed_yearly['proc_mean_aqi_fixed'] - mean_base) / std_base

print("\n=== Fixed processed annual means with z-scores ===")
print(aa_fixed_yearly)

print("\nZ-scores for 2024 & 2025 after fix:")
print(aa_fixed_yearly[aa_fixed_yearly['Year'].isin([2024, 2025])])


Processed (citywide) Annual Average AQI per year:
    Year  proc_mean_aqi
0   2009      18.363369
1   2010      16.914929
2   2011      17.377234
3   2012      15.745745
4   2013      15.222979
5   2014      15.206383
6   2015      14.696335
7   2016      13.678298
8   2017      13.711312
9   2018      13.103546
10  2019      12.166596
11  2020      11.567340
12  2021      11.882687
13  2022      11.062262
14  2023      11.787304
15  2024      46.830601
16  2025      46.053628

Raw citywide annual mean AQI per year:
    Year  raw_mean_aqi
0   2005     54.260274
1   2006     52.115068
2   2007     65.260274
3   2008     61.737705
4   2009     57.410959
5   2010     57.676712
6   2011     59.720548
7   2012     54.297814
8   2013     52.095890
9   2014     51.284932
10  2015     52.271233
11  2016     50.953552
12  2017     49.761644
13  2018     50.789041
14  2019     46.035616
15  2020     43.319672
16  2021     45.841096
17  2022     45.263014
18  2023     49.841096
19  2024     46.83

In [25]:
aa_counts = (
    aq_combined[aq_combined['Time Period'].str.startswith('Annual Average')]
    .groupby('Time Period')
    .size()
    .sort_index()
)

print(aa_counts.loc[['Annual Average 2023', 'Annual Average 2024', 'Annual Average 2025']])


Time Period
Annual Average 2023    282
Annual Average 2024    141
Annual Average 2025    141
dtype: int64


In [26]:
#XOXOXOXOXO

In [28]:
# Work on a copy to be safe
aq_new = aq_new.copy()

# Parse Date
aq_new['Date'] = pd.to_datetime(aq_new['Date'], errors='coerce')

# Drop rows with missing Date or AQI
aq_new = aq_new.dropna(subset=['Date', 'AQI Value'])

# Extract Year
aq_new['Year'] = aq_new['Date'].dt.year

# Keep only years we care about (sane cut)
aq_new = aq_new[aq_new['Year'].between(2005, 2025)]

# Annual citywide mean AQI (simple average across all sites)
raw_yearly = (
    aq_new
    .groupby('Year', as_index=False)['AQI Value']
    .mean()
    .rename(columns={'AQI Value': 'raw_mean_aqi'})
)

print("Raw citywide annual mean AQI:")
print(raw_yearly)

Raw citywide annual mean AQI:
    Year  raw_mean_aqi
0   2005     54.260274
1   2006     52.115068
2   2007     65.260274
3   2008     61.737705
4   2009     57.410959
5   2010     57.676712
6   2011     59.720548
7   2012     54.297814
8   2013     52.095890
9   2014     51.284932
10  2015     52.271233
11  2016     50.953552
12  2017     49.761644
13  2018     50.789041
14  2019     46.035616
15  2020     43.319672
16  2021     45.841096
17  2022     45.263014
18  2023     49.841096
19  2024     46.830601
20  2025     46.053628


In [29]:
# Filter only "Annual Average ..." rows from your original processed file
aa = aq_orig[aq_orig['Time Period'].astype(str).str.startswith('Annual Average')].copy()

# Extract numeric year from "Annual Average YYYY"
aa['Year'] = aa['Time Period'].str.extract(r'(\d{4})').astype(int)

# Citywide mean per year (average of Data Value across all geos)
aa_yearly = (
    aa.groupby('Year', as_index=False)['Data Value']
      .mean()
      .rename(columns={'Data Value': 'proc_mean_aqi'})
)

print("Processed (citywide) Annual Average AQI per year:")
print(aa_yearly)


Processed (citywide) Annual Average AQI per year:
    Year  proc_mean_aqi
0   2009      18.363369
1   2010      16.914929
2   2011      17.377234
3   2012      15.745745
4   2013      15.222979
5   2014      15.206383
6   2015      14.696335
7   2016      13.678298
8   2017      13.711312
9   2018      13.103546
10  2019      12.166596
11  2020      11.567340
12  2021      11.882687
13  2022      11.062262
14  2023      11.787304


In [30]:
# Overlap period where we trust both raw and processed
start_year = 2009
end_year   = 2023

overlap_raw  = raw_yearly[raw_yearly['Year'].between(start_year, end_year)]
overlap_proc = aa_yearly[aa_yearly['Year'].between(start_year, end_year)]

mapping_df = overlap_raw.merge(overlap_proc, on='Year', how='inner')

print("Overlap years used for mapping:")
print(mapping_df)


Overlap years used for mapping:
    Year  raw_mean_aqi  proc_mean_aqi
0   2009     57.410959      18.363369
1   2010     57.676712      16.914929
2   2011     59.720548      17.377234
3   2012     54.297814      15.745745
4   2013     52.095890      15.222979
5   2014     51.284932      15.206383
6   2015     52.271233      14.696335
7   2016     50.953552      13.678298
8   2017     49.761644      13.711312
9   2018     50.789041      13.103546
10  2019     46.035616      12.166596
11  2020     43.319672      11.567340
12  2021     45.841096      11.882687
13  2022     45.263014      11.062262
14  2023     49.841096      11.787304


In [31]:
# Simple linear mapping: proc ≈ m * raw + c
m, c = np.polyfit(mapping_df['raw_mean_aqi'], mapping_df['proc_mean_aqi'], 1)
print(f"Fitted mapping: proc_mean ≈ {m:.4f} * raw_mean + {c:.4f}")


Fitted mapping: proc_mean ≈ 0.4469 * raw_mean + -8.6726


In [32]:
# Get raw means for 2024 & 2025
raw_24_25 = raw_yearly[raw_yearly['Year'].isin([2024, 2025])].copy()

# Apply mapping to get values consistent with processed scale
raw_24_25['pred_proc_mean_aqi'] = m * raw_24_25['raw_mean_aqi'] + c

print("Raw and predicted processed means for 2024 & 2025:")
print(raw_24_25)


Raw and predicted processed means for 2024 & 2025:
    Year  raw_mean_aqi  pred_proc_mean_aqi
19  2024     46.830601           12.255900
20  2025     46.053628           11.908673


In [33]:
aq_final = aq_orig.copy()

# Use Annual Average 2023 as template
aa_2023 = aq_final[aq_final['Time Period'] == 'Annual Average 2023'].copy()
print("Template rows (Annual Average 2023):", aa_2023.shape)


Template rows (Annual Average 2023): (282, 8)


In [34]:
# Create copies for 2024 & 2025
aa_2024 = aa_2023.copy()
aa_2025 = aa_2023.copy()

# Update Time Period and Start_Date
aa_2024['Time Period'] = 'Annual Average 2024'
aa_2025['Time Period'] = 'Annual Average 2025'

aa_2024['Start_Date'] = '2024-01-01'
aa_2025['Start_Date'] = '2025-01-01'

# Get mapped citywide means (rounded optional)
mean_2024 = float(raw_24_25.loc[raw_24_25['Year'] == 2024, 'pred_proc_mean_aqi'].iloc[0])
mean_2025 = float(raw_24_25.loc[raw_24_25['Year'] == 2025, 'pred_proc_mean_aqi'].iloc[0])

# Fill Data Value for all geos for that year with mapped mean
aa_2024['Data Value'] = mean_2024
aa_2025['Data Value'] = mean_2025

print("New block shapes:", aa_2024.shape, aa_2025.shape)


New block shapes: (282, 8) (282, 8)


In [35]:
# Remove any accidental 2024/25 rows from previous experiments
mask_bad = aq_final['Time Period'].isin([
    '2024', '2025',
    'Annual Average 2024', 'Annual Average 2025'
])
aq_final = aq_final[~mask_bad].copy()

print("Shape after dropping old 2024/25 rows (if any):", aq_final.shape)


Shape after dropping old 2024/25 rows (if any): (18862, 8)


In [36]:
# Append clean 2024 & 2025 Annual Average rows
aq_final = pd.concat([aq_final, aa_2024, aa_2025], ignore_index=True)

print("Final shape after adding 2024 & 2025:", aq_final.shape)


Final shape after adding 2024 & 2025: (19426, 8)


In [37]:
aa_counts = (
    aq_final[aq_final['Time Period'].str.startswith('Annual Average')]
    .groupby('Time Period')
    .size()
    .sort_index()
)

print("Annual Average row counts per Time Period:")
print(aa_counts)


Annual Average row counts per Time Period:
Time Period
Annual Average 2009    282
Annual Average 2010    282
Annual Average 2011    282
Annual Average 2012    282
Annual Average 2013    282
Annual Average 2014    282
Annual Average 2015    282
Annual Average 2016    282
Annual Average 2017    282
Annual Average 2018    282
Annual Average 2019    282
Annual Average 2020    282
Annual Average 2021    282
Annual Average 2022    282
Annual Average 2023    282
Annual Average 2024    282
Annual Average 2025    282
dtype: int64


In [38]:
aa_fixed = aq_final[aq_final['Time Period'].str.startswith('Annual Average')].copy()
aa_fixed['Year'] = aa_fixed['Time Period'].str.extract(r'(\d{4})').astype(int)

aa_fixed_yearly = (
    aa_fixed.groupby('Year', as_index=False)['Data Value']
            .mean()
            .rename(columns={'Data Value': 'proc_mean_aqi'})
)

baseline = aa_fixed_yearly[aa_fixed_yearly['Year'] <= 2023]['proc_mean_aqi']
mean_base = baseline.mean()
std_base  = baseline.std()

aa_fixed_yearly['z_from_baseline'] = (aa_fixed_yearly['proc_mean_aqi'] - mean_base) / std_base

print("Fixed processed annual means with z-scores:")
print(aa_fixed_yearly)

print("\nZ-scores for 2024 & 2025:")
print(aa_fixed_yearly[aa_fixed_yearly['Year'].isin([2024, 2025])])


Fixed processed annual means with z-scores:
    Year  proc_mean_aqi  z_from_baseline
0   2009      18.363369         1.834587
1   2010      16.914929         1.201540
2   2011      17.377234         1.403593
3   2012      15.745745         0.690542
4   2013      15.222979         0.462065
5   2014      15.206383         0.454812
6   2015      14.696335         0.231893
7   2016      13.678298        -0.213045
8   2017      13.711312        -0.198616
9   2018      13.103546        -0.464243
10  2019      12.166596        -0.873742
11  2020      11.567340        -1.135649
12  2021      11.882687        -0.997826
13  2022      11.062262        -1.356396
14  2023      11.787304        -1.039513
15  2024      12.255900        -0.834711
16  2025      11.908673        -0.986468

Z-scores for 2024 & 2025:
    Year  proc_mean_aqi  z_from_baseline
15  2024      12.255900        -0.834711
16  2025      11.908673        -0.986468


In [39]:
aq_final.to_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Visual Analytics/Team Project/Processed Data/air_quality_processed_with_annual_2024_2025.csv')
print("File saved")

File saved
